# Creación de ModelForm

# Creación de módulo
python manage.py startapp Users

In [ ]:
# Creación de modelo de usuarios
# src\users\models.py
from django.db import models


GENDER = [
    ("female","Femenino"),
    ("male","Masculino"),
    ("other","Otro")
]

# Create your models here.
class User(models.Model):
    name            = models.CharField(max_length=100, blank=False, null=False)
    paternal_name   = models.CharField(max_length=100, blank=False, null=False)
    maternal_name   = models.CharField(max_length=100, blank=True, null=False)
    age             = models.IntegerField(blank=False, null=False)
    email           = models.EmailField(blank=False, null=False, unique=True)
    gender          = models.CharField(max_length=10, choices=GENDER, blank=False, null=False)
    phone           = models.CharField(max_length=20, blank=True, null=True)

In [ ]:
# Creación de modelform
# src\users\forms.py
from django import forms
from .models import User

class UserForm(forms.ModelForm):
    class Meta:
        model = User
        fields = [
            "name",
            "paternal_name",
            "maternal_name",
            "age",
            "email",
            "gender",
            "phone",
        ]
        labels = {
            "name":"Agrega tu Nombre(s)",
            "paternal_name":"Apellido Paterno",
            "maternal_name":"Apellido Matero",
            "age":"Asigna tu edad",
            "gender":"Selecciona tu género",
            "email":"Correo electrónico",
            "phone":"Agrega tu número movil",
        }
        exclude = []

    def clean_age(self, *args, **kwargs):
        age = self.cleaned_data.get("age")
        if age < 18:
            raise forms.ValidationError("Debes ser mayor de 18 años para registrarte.")
        return age
    
    def clean_email(self, *args, **kwargs):
        email = self.cleaned_data.get("email")
        if "@gmail.com" not in email:
            raise forms.ValidationError("El correo necesita ser de dominio @gmail.com")
        return email
    
    def clean_phone(self, *args, **kwargs):
        phone = self.cleaned_data.get("phone")
        if not str(phone).isdigit():
            raise forms.ValidationError("Asigna un número válido.")
        return phone

In [ ]:
# Ajuste de vista
# src\users\views.py
from django.shortcuts import render
from .forms import UserForm
# Create your views here.
def userCreate(request):
    form = UserForm(request.POST or None)
    if form.is_valid():
        form.save()
        print("Usuario registrado")
    return render(request, "users/user_form.html",context={"form":form})

In [ ]:
# URL de módulo 
# src\users\urls.py
from django.urls import path
from .views import userCreate

urlpatterns = [
    path("create/",userCreate),
]


In [ ]:
# Administración del modelo
# src\users\admin.py
from django.contrib import admin

from .models import User

admin.site.register(User)

In [ ]:
# URL del proyecto
# src\config\urls.py
from django.conf import settings
from django.contrib import admin
from django.urls import include, path

urlpatterns = [
    path("up/", include("up.urls")),
    path("", include("pages.urls")),
    path("analytics/", include("analytics.urls")),
    path("users/",include("users.urls")), # <----Agregado
    path("admin/", admin.site.urls),
]
if not settings.TESTING:
    urlpatterns = [
        *urlpatterns,
        path("__debug__/", include("debug_toolbar.urls")),
    ]


In [ ]:
# Configuraciones del proyecto
# src\config\settings.py
# Application definitions
INSTALLED_APPS = [
    "accounts.apps.AccountsConfig",
    "addresses.apps.AddressesConfig",
    "analytics.apps.AnalyticsConfig",
    "billing.apps.BillingConfig",
    "carts.apps.CartsConfig",
    "products.apps.ProductsConfig",
    "order.apps.OrderConfig",
    "pages.apps.PagesConfig",
    "users.apps.UsersConfig", #<---- Agregado
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
]

In [ ]:
# Template
# src\templates\users\user_form.html
<form method="POST" action="">
    <h1>Creación de usuario</h1>
    {% csrf_token %}
    {{ form.as_p }}
    <input type="submit" value="Guardar">
    {% if delete_url %}
        <a href="{{delete_url}}">Eliminar</a>
    {% endif %}
</form> 
